# Dataset Preparation: Call Center Language Switching SLM

**Train datasets:**
1. `Scicom-intl/Call-Center-Language-Switching`
2. `Scicom-intl/Function-Call`
3. `mesolitica/Malaysian-Multiturn-Chat-Assistant`
4. `mesolitica/Malaysian-Speech-Instructions`

**Test set:** Call Center Language Switching - 50 conversations per language

**Classes:**
- Positive: ends with `<|im_end|>` (complete turn)
- Negative: does NOT end with `<|im_end|>` (incomplete/truncated turn)

**Output:** chinidataset Parquet streaming dataset (multipacked, tokenized) for varlen FA training

**Writer:** `chinidataset.ParquetWriter` with native `uint32[]` variable-length arrays

In [ ]:
!pip install git+https://github.com/Scicom-AI-Enterprise-Organization/ChiniDataset.git -q
!pip install datasets huggingface_hub transformers -q
!pip install multiprocess tqdm pandas numpy -q

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
from datasets import load_dataset
from collections import Counter, defaultdict
from transformers import AutoTokenizer
from chinidataset import ParquetWriter, StreamingDataset
from multiprocess import Pool
import itertools

random.seed(42)
np.random.seed(42)

## Step 1: Download & Explore Datasets

In [ ]:
ds_call_center = load_dataset('Scicom-intl/Call-Center-Language-Switching')
print('Call-Center-Language-Switching:')
print(ds_call_center)
print(ds_call_center['train'].column_names)
print(ds_call_center['train'][0])

In [ ]:
ds_function_call = load_dataset('Scicom-intl/Function-Call')
print('Function-Call:')
print(ds_function_call)
print(ds_function_call['train'].column_names)
print(ds_function_call['train'][0])

In [ ]:
ds_multiturn = load_dataset('mesolitica/Malaysian-Multiturn-Chat-Assistant')
print('Malaysian-Multiturn-Chat-Assistant:')
print(ds_multiturn)
print(ds_multiturn['train'].column_names)
print(ds_multiturn['train'][0])

In [ ]:
ds_speech = load_dataset('mesolitica/Malaysian-Speech-Instructions')
print('Malaysian-Speech-Instructions:')
print(ds_speech)
print(ds_speech['train'].column_names)
print(ds_speech['train'][0])

## Step 2: Unified Conversation Format

Convert all datasets to a unified format:
```python
{
    'messages': [
        {'role': 'system', 'content': '...'},
        {'role': 'user', 'content': '...'},
        {'role': 'assistant', 'content': '...'},
        ...
    ],
    'language': 'malay',  # optional
    'source': 'call-center-language-switching'
}
```

**IMPORTANT:** After running Step 1, inspect each dataset's schema and adjust the conversion functions below accordingly.

In [ ]:
def convert_messages_format(row):
    """
    Handle datasets with 'messages' field in OpenAI format:
    [{'role': 'system', 'content': '...'}, ...]
    """
    if 'messages' in row:
        return row['messages']
    return None


def convert_sharegpt_format(row):
    """
    Handle datasets with 'conversations' field in ShareGPT format:
    [{'from': 'system', 'value': '...'}, {'from': 'human', 'value': '...'}, ...]
    """
    role_map = {'system': 'system', 'human': 'user', 'gpt': 'assistant', 'user': 'user', 'assistant': 'assistant'}
    if 'conversations' in row:
        messages = []
        for turn in row['conversations']:
            role = role_map.get(turn.get('from', turn.get('role', '')), 'user')
            content = turn.get('value', turn.get('content', ''))
            messages.append({'role': role, 'content': content})
        return messages
    return None


def convert_to_unified(row, source_name):
    """
    Try multiple format converters. Adjust priority based on dataset inspection.
    """
    messages = convert_messages_format(row)
    if messages is None:
        messages = convert_sharegpt_format(row)
    if messages is None:
        return None
    return {
        'messages': messages,
        'language': row.get('language', row.get('lang', 'unknown')),
        'source': source_name,
    }

In [ ]:
# Convert all datasets to unified format
# Adjust the conversion logic below after inspecting each dataset's actual schema

all_data = []

print('Converting Call-Center-Language-Switching...')
for row in tqdm(ds_call_center['train']):
    unified = convert_to_unified(row, 'call-center-language-switching')
    if unified:
        all_data.append(unified)

print('Converting Function-Call...')
for row in tqdm(ds_function_call['train']):
    unified = convert_to_unified(row, 'function-call')
    if unified:
        all_data.append(unified)

print('Converting Malaysian-Multiturn-Chat-Assistant...')
for row in tqdm(ds_multiturn['train']):
    unified = convert_to_unified(row, 'multiturn-chat')
    if unified:
        all_data.append(unified)

print('Converting Malaysian-Speech-Instructions...')
for row in tqdm(ds_speech['train']):
    unified = convert_to_unified(row, 'speech-instructions')
    if unified:
        all_data.append(unified)

print(f'\nTotal unified conversations: {len(all_data)}')
source_counts = Counter(d['source'] for d in all_data)
for src, cnt in source_counts.items():
    print(f'  {src}: {cnt}')

## Step 3: Build Test Set

From **Call-Center-Language-Switching** only: sample 50 conversations per language.

In [ ]:
# Group call center data by language
call_center_data = [d for d in all_data if d['source'] == 'call-center-language-switching']
by_language = defaultdict(list)
for d in call_center_data:
    by_language[d['language']].append(d)

print('Languages found:')
for lang, items in by_language.items():
    print(f'  {lang}: {len(items)} conversations')

# Sample 50 per language for test set
test_set = []
test_indices = set()

for lang, items in by_language.items():
    n_sample = min(50, len(items))
    sampled = random.sample(items, n_sample)
    for s in sampled:
        s['split'] = 'test'
        test_set.append(s)
        test_indices.add(id(s))
    print(f'  Test: {lang} -> {n_sample} conversations')

print(f'\nTotal test set: {len(test_set)} conversations')

# Remove test samples from training data
train_data = [d for d in all_data if id(d) not in test_indices]
print(f'Total train set: {len(train_data)} conversations')

## Step 4: Chat Template Formatting + Positive/Negative Classes

**Qwen3 chat template:**
```
<|im_start|>system
{system_message}<|im_end|>
<|im_start|>user
{user_message}<|im_end|>
<|im_start|>assistant
{assistant_message}<|im_end|>
```

- **Positive class:** full conversation ending with `<|im_end|>`
- **Negative class:** truncated - last `<|im_end|>` removed, optionally cut at random point in last assistant response

In [ ]:
def format_chat_template(messages):
    """
    Format messages into Qwen3 chat template string.
    Returns the full formatted string ending with <|im_end|>.
    """
    parts = []
    for msg in messages:
        role = msg['role']
        content = msg['content']
        parts.append(f'<|im_start|>{role}\n{content}<|im_end|>')
    return '\n'.join(parts)


def build_positive(messages):
    """
    Positive class: complete conversation ending with <|im_end|>.
    """
    return format_chat_template(messages)


def build_negative(messages):
    """
    Negative class: truncated conversation NOT ending with <|im_end|>.
    Strategy: remove the last <|im_end|> and optionally truncate
    some of the last assistant response.
    """
    text = format_chat_template(messages)
    # Remove the last <|im_end|>
    if text.endswith('<|im_end|>'):
        text = text[:-len('<|im_end|>')]
    
    # Optionally truncate some of the last response (50% chance)
    if random.random() < 0.5 and len(text) > 100:
        # Find the last assistant content and truncate at random point
        last_assistant_idx = text.rfind('<|im_start|>assistant\n')
        if last_assistant_idx >= 0:
            content_start = last_assistant_idx + len('<|im_start|>assistant\n')
            content = text[content_start:]
            if len(content) > 10:
                # Keep 20-80% of the last response
                keep_ratio = random.uniform(0.2, 0.8)
                keep_len = max(5, int(len(content) * keep_ratio))
                text = text[:content_start + keep_len]
    
    return text.rstrip()


# Quick test
test_messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': 'Hello, boleh tolong saya?'},
    {'role': 'assistant', 'content': 'Sudah tentu! Saya sedia membantu. Apa yang anda perlukan?'},
]
print('=== POSITIVE ===')
print(repr(build_positive(test_messages)))
print()
print('=== NEGATIVE ===')
print(repr(build_negative(test_messages)))

In [ ]:
# Build positive and negative samples for training
train_samples = []

for d in tqdm(train_data, desc='Building train samples'):
    messages = d['messages']
    if not messages or len(messages) < 2:
        continue
    
    # Positive: full conversation
    pos_text = build_positive(messages)
    train_samples.append({'text': pos_text, 'label': 'positive', 'source': d['source']})
    
    # Negative: truncated conversation
    neg_text = build_negative(messages)
    train_samples.append({'text': neg_text, 'label': 'negative', 'source': d['source']})

random.shuffle(train_samples)

print(f'Total train samples: {len(train_samples)}')
label_counts = Counter(s['label'] for s in train_samples)
print(f'  Positive: {label_counts["positive"]}')
print(f'  Negative: {label_counts["negative"]}')

In [ ]:
# Build positive and negative samples for test set
test_samples = []

for d in tqdm(test_set, desc='Building test samples'):
    messages = d['messages']
    if not messages or len(messages) < 2:
        continue
    
    pos_text = build_positive(messages)
    test_samples.append({'text': pos_text, 'label': 'positive', 'language': d['language']})
    
    neg_text = build_negative(messages)
    test_samples.append({'text': neg_text, 'label': 'negative', 'language': d['language']})

print(f'Total test samples: {len(test_samples)}')
for lang in set(s['language'] for s in test_samples):
    lang_samples = [s for s in test_samples if s['language'] == lang]
    print(f'  {lang}: {len(lang_samples)} (pos={sum(1 for s in lang_samples if s["label"]=="positive")}, neg={sum(1 for s in lang_samples if s["label"]=="negative")})')

## Step 5: Tokenize & Write Parquet (Multipacking)

Using `chinidataset.ParquetWriter` with native `uint32[]` variable-length arrays.

Pack multiple tokenized conversations into blocks of `BLOCK_SIZE` tokens.
Each row = one packed block with:
- `input_ids` (`uint32[]`): concatenated token IDs
- `position_ids` (`uint32[]`): per-sequence position indices (reset per conversation)
- `attention_mask` (`uint32[]`): array of sequence lengths (for varlen FA cu_seqlens)
- `text` (`str`): metadata

In [ ]:
# Setup tokenizer - use the same base model tokenizer for all scales
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-0.6B-Base')
print(f'Vocab size: {len(tokenizer)}')

# Verify chat template tokens exist
im_start_id = tokenizer.convert_tokens_to_ids('<|im_start|>')
im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')
print(f'<|im_start|> token ID: {im_start_id}')
print(f'<|im_end|> token ID: {im_end_id}')

# Test tokenization
test_tok = tokenizer(build_positive(test_messages), add_special_tokens=False)
print(f'Test tokenization length: {len(test_tok["input_ids"])}')
# Verify last token is <|im_end|>
print(f'Last token: {test_tok["input_ids"][-1]} (expected {im_end_id})')

In [ ]:
# chinidataset ParquetWriter schema
# uint32[] = variable-length uint32 array (native support, no custom encoding needed)

columns = {
    'input_ids': 'uint32[]',
    'position_ids': 'uint32[]',
    'attention_mask': 'uint32[]',
    'text': 'str',
}


def collator_pack(batch_ids, batch_position_ids):
    """
    Combine multiple tokenized sequences into a single packed row.
    attention_mask stores sequence lengths (not 0/1 mask) for varlen FA cu_seqlens.
    """
    input_ids = []
    position_ids = []
    masks = []
    for i in range(len(batch_ids)):
        l = len(batch_ids[i])
        input_ids.extend(batch_ids[i])
        position_ids.extend(batch_position_ids[i])
        masks.append(l)
    return {
        'input_ids': np.array(input_ids, dtype=np.uint32),
        'position_ids': np.array(position_ids, dtype=np.uint32),
        'attention_mask': np.array(masks, dtype=np.uint32),
        'text': '',
    }

In [ ]:
def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i:i+n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()
    if returned:
        return list(itertools.chain(*pooled))


BLOCK_SIZE = 8192  # tokens per packed block
OUTPUT_DIR = './parquet-train'


def write_parquet_worker(args):
    """
    Worker: tokenize samples, multipack into blocks, write with ParquetWriter.
    """
    rows, index = args
    out_root = f'{OUTPUT_DIR}/shard-{index}'
    
    count = 0
    temp_ids = []
    temp_pos = []
    written = 0
    
    with ParquetWriter(out=out_root, columns=columns, exist_ok=True) as writer:
        for row in tqdm(rows, desc=f'Worker {index}'):
            text = row['text']
            outputs = tokenizer(text, add_special_tokens=False)
            ids = outputs['input_ids']
            position = list(range(len(ids)))
            length = len(ids)
            
            if length == 0:
                continue
            
            # Skip sequences longer than block_size
            if length > BLOCK_SIZE:
                continue
            
            if count + length > BLOCK_SIZE:
                # Write current packed block
                o = collator_pack(temp_ids, temp_pos)
                if o['input_ids'].shape[0] > 0:
                    writer.write(o)
                    written += 1
                # Start new block
                temp_ids = [ids]
                temp_pos = [position]
                count = length
            else:
                temp_ids.append(ids)
                temp_pos.append(position)
                count += length
        
        # Write remaining
        if len(temp_ids) > 0:
            o = collator_pack(temp_ids, temp_pos)
            if o['input_ids'].shape[0] > 0:
                writer.write(o)
                written += 1
    
    return [written]

In [ ]:
# Write training parquet with multiprocessing
os.makedirs(OUTPUT_DIR, exist_ok=True)

N_WORKERS = 20  # adjust based on CPU cores
results = multiprocessing(train_samples, write_parquet_worker, cores=N_WORKERS)
print(f'Total blocks written across all workers: {sum(results)}')

In [ ]:
# Merge all shards into single dataset
shard_folders = sorted(
    glob(f'{OUTPUT_DIR}/shard-*'),
    key=lambda x: int(x.split('-')[-1])
)
print(f'Found {len(shard_folders)} shards')

MERGED_DIR = './parquet-train-merged'

total = 0
with ParquetWriter(out=MERGED_DIR, columns=columns, exist_ok=True) as writer:
    for folder in shard_folders:
        try:
            ds = StreamingDataset(local=folder)
            for i in tqdm(range(len(ds)), desc=folder):
                writer.write(ds[i])
                total += 1
        except Exception as e:
            print(f'Error reading {folder}: {e}')

print(f'\nMerged dataset: {total} blocks written to {MERGED_DIR}')

## Step 6: Write Test Set Parquet

In [ ]:
# Test set: each sample is its own row (no multipacking)
# so we can evaluate per-sample

TEST_DIR = './parquet-test'

total_test = 0
with ParquetWriter(out=TEST_DIR, columns=columns, exist_ok=True) as writer:
    for sample in tqdm(test_samples, desc='Writing test parquet'):
        text = sample['text']
        outputs = tokenizer(text, add_special_tokens=False)
        ids = outputs['input_ids']
        if len(ids) == 0:
            continue
        
        row = {
            'input_ids': np.array(ids, dtype=np.uint32),
            'position_ids': np.array(list(range(len(ids))), dtype=np.uint32),
            'attention_mask': np.array([len(ids)], dtype=np.uint32),
            'text': json.dumps({'label': sample['label'], 'language': sample['language']}),
        }
        writer.write(row)
        total_test += 1

print(f'Test set: {total_test} samples written to {TEST_DIR}')

## Step 7: Verify Dataset

In [ ]:
# Verify train dataset
ds_train = StreamingDataset(local=MERGED_DIR)
print(f'Train dataset: {len(ds_train)} packed blocks')

# Inspect first block
sample = ds_train[0]
print(f'Keys: {list(sample.keys())}')
print(f'input_ids shape: {sample["input_ids"].shape}')
print(f'position_ids shape: {sample["position_ids"].shape}')
print(f'attention_mask (seq lengths): {sample["attention_mask"]}')
print(f'Number of sequences in block: {len(sample["attention_mask"])}')
print(f'Total tokens in block: {sample["input_ids"].shape[0]}')

# Decode first sequence in first block
first_seq_len = int(sample['attention_mask'][0])
first_seq_ids = sample['input_ids'][:first_seq_len]
decoded = tokenizer.decode(first_seq_ids)
print(f'\nFirst sequence ({first_seq_len} tokens):')
print(decoded[:500])
print(f'\n... ends with <|im_end|>: {decoded.rstrip().endswith("<|im_end|>")}')

In [ ]:
# Verify test dataset
ds_test = StreamingDataset(local=TEST_DIR)
print(f'Test dataset: {len(ds_test)} samples')

# Check distribution
pos_count = 0
neg_count = 0
lang_dist = Counter()
for i in range(len(ds_test)):
    meta = json.loads(ds_test[i]['text'])
    if meta['label'] == 'positive':
        pos_count += 1
    else:
        neg_count += 1
    lang_dist[meta['language']] += 1

print(f'Positive: {pos_count}, Negative: {neg_count}')
print(f'By language: {dict(lang_dist)}')

In [ ]:
# Verify positive samples end with <|im_end|> and negative don't
im_end_id = tokenizer.convert_tokens_to_ids('<|im_end|>')

correct = 0
total = len(ds_test)
for i in range(total):
    sample = ds_test[i]
    meta = json.loads(sample['text'])
    ids = sample['input_ids']
    last_token = int(ids[-1])
    
    if meta['label'] == 'positive' and last_token == im_end_id:
        correct += 1
    elif meta['label'] == 'negative' and last_token != im_end_id:
        correct += 1

print(f'Label verification: {correct}/{total} correct ({100*correct/total:.1f}%)')

## Step 8: Upload to HuggingFace

In [ ]:
# Upload train dataset
!huggingface-cli upload Scicom-intl/call-center-ls-multipacking-train {MERGED_DIR} --repo-type=dataset --private

# Upload test dataset
!huggingface-cli upload Scicom-intl/call-center-ls-test {TEST_DIR} --repo-type=dataset --private

## Stats Summary

In [ ]:
# Token statistics
total_tokens = 0
for i in tqdm(range(len(ds_train)), desc='Counting tokens'):
    total_tokens += ds_train[i]['input_ids'].shape[0]

print(f'\n=== Dataset Summary ===')
print(f'Train blocks: {len(ds_train)}')
print(f'Train total tokens: {total_tokens:,}')
print(f'Train samples (positive + negative): {len(train_samples)}')
print(f'Test samples: {len(test_samples)}')
print(f'Block size: {BLOCK_SIZE}')
print(f'\nReady for training with:')
print(f'  python train.py --train_file {MERGED_DIR} --block_size {BLOCK_SIZE}')